# Notebook 1: Baseline Semantic BERTopic Clustering
---
## Strategy
**Pure unsupervised semantic discovery** using BERTopic with LaBSE embeddings.
- No manual topic seeding
- No entity normalization
- HDBSCAN finds natural density-based clusters

## Flow
1. Load raw video titles
2. Clean text (remove URLs, hashtags, channel codes)
3. Generate LaBSE embeddings (768-dim, multilingual)
4. UMAP reduces to 5 dimensions
5. HDBSCAN groups by density
6. BERTopic extracts keywords per cluster

## When to Use
- When you have NO idea what topics exist in your data
- For exploratory analysis of new datasets
- As a baseline to compare other methods against

## Limitations
- High noise ratio (~25%) because HDBSCAN is strict
- Same person (Modi/मोदी) can split into multiple clusters


In [ ]:
# Install Dependencies
!pip install -q bertopic sentence-transformers hdbscan umap-learn pandas scikit-learn
print("Dependencies installed.")

In [ ]:
import pandas as pd, numpy as np, re, html
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from google.colab import files

# Upload your data file
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Loaded: {filename}")

## Text Cleaning Engine
Strips noise while preserving Devanagari (Hindi) script.

In [ ]:
DOMAIN_STOPS = {"abp","amp","breaking","channel","click","exclusive","facebook","follow","hindi",
    "http","https","instagram","latest","live","news","shorts","subscribe","today",
    "video","videos","watch","viral","youtube","aajtak","ndtv","at2","n18v","tv9"}

def normalize_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    tokens = [t for t in text.lower().split()
             if not (t.isascii() and len(t) <= 2) and t not in DOMAIN_STOPS]
    return " ".join(tokens).strip()

# --- UNIVERSAL DATA LOADER ---
import pandas as pd
import os

# Get the file extension
file_ext = os.path.splitext(filename)[1].lower()

if file_ext == '.xlsx':
    print("📂 Detected Excel file. Using read_excel...")
    df = pd.read_excel(filename)
elif file_ext == '.csv' or file_ext == '.txt':
    print("📂 Detected Text/CSV file. Using read_csv with encoding recovery...")
    try:
        df = pd.read_csv(filename, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(filename, encoding='latin1')
        except:
            df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')
else:
    print(f"⚠️ Unknown format {file_ext}. Attempting generic read...")
    df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')

# Ensure we have a title column
if "title" not in df.columns:
    if len(df.columns) == 1:
        df.columns = ["title"]
    else:
        df.columns = ["title"] + list(df.columns[1:])
# ------------------------------
df["clean_text"] = df["title"].fillna("").astype(str).map(normalize_text)
df = df[df["clean_text"] != ""].reset_index(drop=True)
print(f"Cleaned {len(df)} documents")

## Embedding + Clustering

In [ ]:
embedder = SentenceTransformer("sentence-transformers/LaBSE")
embeddings = embedder.encode(df["clean_text"].tolist(), show_progress_bar=True, batch_size=32)

model = BERTopic(
    embedding_model=embedder,
    min_topic_size=15,
    nr_topics="auto",
    verbose=True
)
topics, probs = model.fit_transform(df["clean_text"].tolist(), embeddings)
df["topic_id"] = topics

noise_pct = round(100 * (df["topic_id"] == -1).mean(), 1)
print(f"Topics: {len(set(topics)) - 1}, Noise: {noise_pct}%")

## Results

In [ ]:
topic_info = model.get_topic_info()
topic_info.head(20)

In [ ]:
## FINAL CLEAN STANDARDIZED EXPORT
# 1. Create a CLEAN Topic Label Map (No numbers, Space-separated, Title-case)
topic_info = model.get_topic_info()
label_map = {}
for _, row in topic_info.iterrows():
    t_id = row['Topic']
    raw_name = row['Name']
    # Remove the "0_" or "-1_" prefix and clean up underscores
    if "_" in raw_name:
        clean_name = raw_name.split("_", 1)[-1].replace("_", " ").title()
    else:
        clean_name = raw_name.title()
    label_map[t_id] = clean_name

df['topic_label'] = df['topic_id'].map(label_map)

# 2. Reorganize Columns
if 'ner_text' not in df.columns: df['ner_text'] = df['clean_text']
final_df = df.rename(columns={'title': 'raw_text'})

# 3. Selection (Exactly 5 columns)
cols_to_keep = ['raw_text', 'clean_text', 'ner_text', 'topic_id', 'topic_label']
final_df = final_df[cols_to_keep]

# 4. Save and Download
output_fn = "final_clustering_results.csv"
final_df.to_csv(output_fn, index=False)
print(f"Success! Exported {len(final_df)} titles with clean, unique labels.")
from google.colab import files
files.download(output_fn)